# Binary baseline and supplemental comparison

Reconstructed and executed during snapshot preparation. RF settings reproduce recorded code. Logistic Regression and Gradient Boosting results below are supplemental executions now, not original Weeks 5–6 logs. No calibration is fitted.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load_matches, prepare_binary, dataset_audit, chronological_split, BASE_FEATURES
from src.features import add_rolling_features, verify_rolling_features, ROLLING_FEATURES
from src.experiments import models, evaluate


In [2]:
data = prepare_binary(load_matches())
complete = add_rolling_features(data).dropna(subset=ROLLING_FEATURES).reset_index(drop=True)
print("Target: team win=1; draw/loss=0")
print("Raw rows:", len(data), "complete rolling rows:", len(complete))

Target: team win=1; draw/loss=0
Raw rows: 1389 complete rolling rows: 1317


In [3]:
result = evaluate(models()["Random Forest"], data, BASE_FEATURES)
print(json.dumps(result, indent=2))

{
  "train_rows": 1107,
  "test_rows": 276,
  "train_last_date": "2021-12-30",
  "test_first_date": "2022-01-02",
  "excluded_cutoff_day_rows": 6,
  "accuracy": 0.6123188405797102,
  "win_precision": 0.4745762711864407,
  "win_recall": 0.2692307692307692,
  "win_f1": 0.34355828220858897,
  "macro_f1": 0.534247007428202,
  "binary_log_loss": 0.6793711504939138,
  "binary_brier": 0.2383456068209675,
  "confusion_matrix_actual_rows_predicted_columns": [
    [
      141,
      31
    ],
    [
      76,
      28
    ]
  ],
  "always_non_win_accuracy": 0.6231884057971014,
  "features": [
    "venue_code",
    "opp_code",
    "hour",
    "day_code"
  ]
}


The recorded baseline achieved 61.23% accuracy, below the 62.32% always-non-win baseline. Accuracy alone is not enough. The final project must evaluate three-class probabilities and calibration separately.

In [4]:
rolling_result = evaluate(models()["Random Forest"], complete, BASE_FEATURES + ROLLING_FEATURES)
print(json.dumps(rolling_result, indent=2))

{
  "train_rows": 1035,
  "test_rows": 276,
  "train_last_date": "2021-12-30",
  "test_first_date": "2022-01-02",
  "excluded_cutoff_day_rows": 6,
  "accuracy": 0.6666666666666666,
  "win_precision": 0.625,
  "win_recall": 0.28846153846153844,
  "win_f1": 0.39473684210526316,
  "macro_f1": 0.5823684210526316,
  "binary_log_loss": 0.620971946335985,
  "binary_brier": 0.21602794765529612,
  "confusion_matrix_actual_rows_predicted_columns": [
    [
      154,
      18
    ],
    [
      74,
      30
    ]
  ],
  "always_non_win_accuracy": 0.6231884057971014,
  "features": [
    "venue_code",
    "opp_code",
    "hour",
    "day_code",
    "gf_rolling",
    "ga_rolling",
    "sh_rolling",
    "sot_rolling",
    "dist_rolling",
    "fk_rolling",
    "pk_rolling",
    "pkatt_rolling"
  ]
}


Recorded rolling **precision** was 0.625. The complete score set above is from this new execution. Eligible rows differ between raw and rolling runs; a final ablation must use the same fixtures.

In [5]:
for name, estimator in models().items():
    row = evaluate(estimator, complete, BASE_FEATURES + ROLLING_FEATURES)
    print(f"{name:20} accuracy={row['accuracy']:.4f} win_precision={row['win_precision']:.4f} log_loss={row['binary_log_loss']:.4f}")

Random Forest        accuracy=0.6667 win_precision=0.6250 log_loss=0.6210
Logistic Regression  accuracy=0.6159 win_precision=0.4583 log_loss=0.6466
Gradient Boosting    accuracy=0.6630 win_precision=0.5902 log_loss=0.6331


## Next stage

Do not use these exploratory test scores to claim a calibrated final system. See `docs/METHODOLOGY.md` for independent tuning/calibration/final testing, three-class metrics, controlled feature ablation, and interpretation.